<a href="https://colab.research.google.com/github/aditiprashant07/Deforestation-Detection/blob/main/01_GEE_Sentinel2_Export.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q earthengine-api geemap


In [ ]:
import ee
import geemap

# 1. Trigger the authentication flow
ee.Authenticate()

# 2. Initialize with your Cloud Project ID (found at the top right of your GEE Code Editor)
# Replace 'your-project-id' with your actual Google Cloud Project ID
ee.Initialize(project='project-26a52c32-ca22-486c-86d')

In [ ]:
# Rondônia, Brazil (Latitude: ~ -10.5, Longitude: ~ -62.8)
aoi = ee.Geometry.Rectangle([-62.85, -10.55, -62.75, -10.45])

# Switch basemap to satellite imagery (e.g., 'HYBRID' or 'SATELLITE')
Map = geemap.Map(basemap='HYBRID')
Map.centerObject(aoi, zoom=12)
Map.addLayer(aoi, {'color': 'red'}, 'AOI Boundary')
Map

In [ ]:
def mask_s2_clouds(image):
    qa = image.select('QA60')
    # Bits 10 and 11 represent clouds and cirrus
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
        qa.bitwiseAnd(cirrus_bit_mask).eq(0)
    )
    # Scale reflectance to 0-1 range and retain bands B2, B3, B4, B8
    return image.updateMask(mask).select(['B2', 'B3', 'B4', 'B8'])

# 1. Baseline Image T1 (July-August 2022)
s2_t1 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filterDate('2022-07-01', '2022-08-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .map(mask_s2_clouds)
    .median()
    .clip(aoi)
)

# 2. Recent Image T2 (July-August 2024)
s2_t2 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filterDate('2024-07-01', '2024-08-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .map(mask_s2_clouds)
    .median()
    .clip(aoi)
)

# Visualize True Color (B4=Red, B3=Green, B2=Blue)
vis_params = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}
Map.addLayer(s2_t1, vis_params, 'Sentinel-2 2022 (T1)')
Map.addLayer(s2_t2, vis_params, 'Sentinel-2 2024 (T2)')
Map

In [ ]:
# Task 1: Export T1 (2022)
task_t1 = ee.batch.Export.image.toDrive(
    image=s2_t1,
    description='Sentinel2_Rondonia_2022',
    folder='Deforestation_Project',
    fileNamePrefix='s2_2022_b2_b3_b4_b8',
    region=aoi,
    scale=10,  # 10 meter resolution
    crs='EPSG:4326',
    maxPixels=1e9
)
task_t1.start()

# Task 2: Export T2 (2024)
task_t2 = ee.batch.Export.image.toDrive(
    image=s2_t2,
    description='Sentinel2_Rondonia_2024',
    folder='Deforestation_Project',
    fileNamePrefix='s2_2024_b2_b3_b4_b8',
    region=aoi,
    scale=10,
    crs='EPSG:4326',
    maxPixels=1e9
)
task_t2.start()

print("Export tasks initiated! Check status via task_t1.status() or the Tasks tab in the GEE Code Editor.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q rasterio matplotlib

In [ ]:
import rasterio

# Update path if your folder name is different
path_2022 = '/content/drive/MyDrive/Deforestation_Project/s2_2022_b2_b3_b4_b8.tif'
path_2024 = '/content/drive/MyDrive/Deforestation_Project/s2_2024_b2_b3_b4_b8.tif'

with rasterio.open(path_2022) as src_22, rasterio.open(path_2024) as src_24:
    print("--- 2022 Metadata ---")
    print(f"Dimensions: {src_22.width} x {src_22.height}, Bands: {src_22.count}")
    print(f"CRS: {src_22.crs}")
    print(f"Bounds: {src_22.bounds}")

    print("\n--- 2024 Metadata ---")
    print(f"Dimensions: {src_24.width} x {src_24.height}, Bands: {src_24.count}")
    print(f"CRS: {src_24.crs}")
    print(f"Bounds: {src_24.bounds}")

    # Assert alignment
    assert src_22.crs == src_24.crs, "CRS mismatch!"
    assert src_22.shape == src_24.shape, "Image shape mismatch!"
    print("\n Alignment verified successfully!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def read_and_normalize(path):
    with rasterio.open(path) as src:
        # rasterio bands are 1-indexed: 1=Blue, 2=Green, 3=Red, 4=NIR
        blue = src.read(1).astype(np.float32)
        green = src.read(2).astype(np.float32)
        red = src.read(3).astype(np.float32)
        nir = src.read(4).astype(np.float32)

        # Normalize 0-3000 reflectance range to [0, 1] for display
        def norm(b):
            return np.clip(b / 3000.0, 0, 1)

        # True-Color: RGB (Red, Green, Blue)
        rgb = np.dstack([norm(red), norm(green), norm(blue)])

        # False-Color (Infrared): NIR, Red, Green
        # Dense vegetation turns bright red; cleared soil turns gray/cyan
        false_color = np.dstack([norm(nir), norm(red), norm(green)])

        return rgb, false_color

rgb_22, fc_22 = read_and_normalize(path_2022)
rgb_24, fc_24 = read_and_normalize(path_2024)

# Plot Comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 14))

axes[0, 0].imshow(rgb_22)
axes[0, 0].set_title("2022 True Color (RGB)", fontsize=13)
axes[0, 0].axis('off')

axes[0, 1].imshow(rgb_24)
axes[0, 1].set_title("2024 True Color (RGB)", fontsize=13)
axes[0, 1].axis('off')

axes[1, 0].imshow(fc_22)
axes[1, 0].set_title("2022 False Color (NIR / Red / Green)", fontsize=13)
axes[1, 0].axis('off')

axes[1, 1].imshow(fc_24)
axes[1, 1].set_title("2024 False Color (NIR / Red / Green)", fontsize=13)
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
def compute_ndvi(path):
    with rasterio.open(path) as src:
        red = src.read(3).astype(np.float32)
        nir = src.read(4).astype(np.float32)
        # Avoid division by zero
        ndvi = (nir - red) / (nir + red + 1e-6)
        return ndvi

ndvi_22 = compute_ndvi(path_2022)
ndvi_24 = compute_ndvi(path_2024)

# Deforestation = High NDVI in 2022 dropping significantly in 2024
ndvi_diff = ndvi_22 - ndvi_24

plt.figure(figsize=(8, 7))
plt.imshow(ndvi_diff, cmap='RdYlGn_r', vmin=0, vmax=0.5)
plt.colorbar(label='NDVI Drop (Deforestation Intensity)')
plt.title("Vegetation Loss: 2022 vs 2024", fontsize=14)
plt.axis('off')
plt.show()

In [ ]:
!pip install -q segmentation-models-pytorch rasterio

In [ ]:
import os
import rasterio
import numpy as np

# Ensure destination folders exist
os.makedirs("/content/patches/images", exist_ok=True)
os.makedirs("/content/patches/masks", exist_ok=True)

def process_and_tile(raster_path, prefix="2022", patch_size=256):
    with rasterio.open(raster_path) as src:
        # Read bands: 1=B2 (Blue), 2=B3 (Green), 3=B4 (Red), 4=B8 (NIR)
        img = src.read().astype(np.float32)  # Shape: (4, H, W)

    # Reorder dimensions to (H, W, Channels)
    img = np.moveaxis(img, 0, -1)

    # Normalize reflectance values (0 - 3000 -> 0.0 - 1.0)
    img_norm = np.clip(img / 3000.0, 0.0, 1.0)

    # Calculate NDVI: (NIR - Red) / (NIR + Red)
    red = img[:, :, 2]
    nir = img[:, :, 3]
    ndvi = (nir - red) / (nir + red + 1e-6)

    # Binary forest mask: 1 = Forest (NDVI > 0.6), 0 = Cleared / Non-forest
    mask = (ndvi > 0.6).astype(np.uint8)

    # Calculate valid dimension bounds that fit whole multiples of patch_size
    h, w, c = img_norm.shape
    crop_h = (h // patch_size) * patch_size
    crop_w = (w // patch_size) * patch_size

    patch_id = 0
    # Pure NumPy window slicing
    for y in range(0, crop_h, patch_size):
        for x in range(0, crop_w, patch_size):
            single_img = img_norm[y:y+patch_size, x:x+patch_size, :]
            single_mask = mask[y:y+patch_size, x:x+patch_size]

            # Save patch pairs
            np.save(f"/content/patches/images/{prefix}_img_{patch_id}.npy", single_img)
            np.save(f"/content/patches/masks/{prefix}_mask_{patch_id}.npy", single_mask)
            patch_id += 1

    print(f"Successfully saved {patch_id} patches for {prefix}")

# File paths to your Google Drive rasters
path_2022 = '/content/drive/MyDrive/Deforestation_Project/s2_2022_b2_b3_b4_b8.tif'
path_2024 = '/content/drive/MyDrive/Deforestation_Project/s2_2024_b2_b3_b4_b8.tif'

process_and_tile(path_2022, prefix="2022")
process_and_tile(path_2024, prefix="2024")

In [ ]:
import os
import glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split

class ForestDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_paths = sorted(glob.glob(f"{img_dir}/*.npy"))
        self.mask_paths = sorted(glob.glob(f"{mask_dir}/*.npy"))

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        # Load image (256, 256, 4) and mask (256, 256)
        img = np.load(self.img_paths[idx])
        mask = np.load(self.mask_paths[idx])

        # Convert (H, W, Channels) -> PyTorch format (Channels, H, W)
        img = np.moveaxis(img, -1, 0)
        mask = np.expand_dims(mask, axis=0)  # Shape: (1, 256, 256)

        return torch.tensor(img, dtype=torch.float32), torch.tensor(mask, dtype=torch.float32)

# Load full dataset
full_dataset = ForestDataset("/content/patches/images", "/content/patches/masks")
total_samples = len(full_dataset)

# Split 80% train, 20% validation
train_size = int(0.8 * total_samples)
val_size = total_samples - train_size
train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

print(f"Total Patches: {total_samples}")
print(f"Training: {len(train_dataset)} | Validation: {len(val_dataset)}")

In [ ]:
import segmentation_models_pytorch as smp

device = "cuda" if torch.cuda.is_available() else "cpu"

# 4-channel input (B2, B3, B4, B8)
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=4,
    classes=1,
    activation=None # We will output raw logits for numerical stability in BCEWithLogitsLoss
).to(device)

# Loss functions
bce_loss = torch.nn.BCEWithLogitsLoss()
dice_loss = smp.losses.DiceLoss(mode="binary")

def combined_loss(pred_logits, targets):
    return bce_loss(pred_logits, targets) + dice_loss(pred_logits, targets)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

print(f"Model and optimizers initialized on: {device}")

In [ ]:
epochs = 15

print("Starting U-Net Training...")
for epoch in range(1, epochs + 1):
    model.train()
    train_loss = 0.0

    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = combined_loss(outputs, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    scheduler.step()
    train_loss /= len(train_dataset)

    # Validation step
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)
            outputs = model(images)
            loss = combined_loss(outputs, masks)
            val_loss += loss.item() * images.size(0)

    val_loss /= len(val_dataset)

    print(f"Epoch [{epoch:02d}/{epochs:02d}] - Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

# Save the trained model weights directly to your Google Drive
save_path = "/content/drive/MyDrive/Deforestation_Project/forest_unet.pth"
torch.save(model.state_dict(), save_path)
print(f"\nModel weights successfully saved to: {save_path}")

In [ ]:
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    sample_img, sample_mask = val_dataset[0]
    input_tensor = sample_img.unsqueeze(0).to(device)

    # Generate prediction probability map
    raw_pred = model(input_tensor)
    prob_mask = torch.sigmoid(raw_pred).squeeze().cpu().numpy()
    binary_pred = (prob_mask > 0.5).astype(np.uint8)

# Convert 4-band image (Channels, H, W) to RGB for display (Bands: 0=Blue, 1=Green, 2=Red, 3=NIR)
rgb = sample_img.numpy().transpose(1, 2, 0)[:, :, [2, 1, 0]]
rgb = np.clip(rgb, 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(rgb)
axes[0].set_title("Input Patch (RGB)")
axes[0].axis("off")

axes[1].imshow(sample_mask.squeeze(), cmap="Greens")
axes[1].set_title("Target Mask (Ground Truth)")
axes[1].axis("off")

axes[2].imshow(binary_pred, cmap="Greens")
axes[2].set_title("U-Net Prediction")
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
!pip install -q geopandas shapely scipy

In [ ]:
import numpy as np
import rasterio
import torch
import segmentation_models_pytorch as smp

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Load the trained U-Net model
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=4,
    classes=1,
    activation="sigmoid"
).to(device)

model_weights_path = "/content/drive/MyDrive/Deforestation_Project/forest_unet.pth"
model.load_state_dict(torch.load(model_weights_path, map_location=device))
model.eval()
print("Trained model weights loaded successfully!")

# 2. Sliding window inference function
def predict_full_raster(raster_path, patch_size=256):
    with rasterio.open(raster_path) as src:
        profile = src.profile
        img = src.read().astype(np.float32)  # Shape: (4, H, W)

    _, h, w = img.shape
    # Reorder to (H, W, Channels) and normalize
    img_norm = np.moveaxis(img, 0, -1)
    img_norm = np.clip(img_norm / 3000.0, 0.0, 1.0)

    # Pad image dimensions to clean multiples of patch_size
    pad_h = (patch_size - (h % patch_size)) % patch_size
    pad_w = (patch_size - (w % patch_size)) % patch_size
    img_padded = np.pad(img_norm, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")

    full_mask = np.zeros((img_padded.shape[0], img_padded.shape[1]), dtype=np.float32)

    with torch.no_grad():
        for y in range(0, img_padded.shape[0], patch_size):
            for x in range(0, img_padded.shape[1], patch_size):
                patch = img_padded[y:y+patch_size, x:x+patch_size, :]
                patch_tensor = torch.tensor(patch).permute(2, 0, 1).unsqueeze(0).float().to(device)

                pred = model(patch_tensor)
                full_mask[y:y+patch_size, x:x+patch_size] = pred.squeeze().cpu().numpy()

    # Crop back to original dimensions and threshold to binary (1 = Forest, 0 = Non-forest)
    binary_mask = (full_mask[:h, :w] > 0.5).astype(np.uint8)
    return binary_mask, profile

path_2022 = '/content/drive/MyDrive/Deforestation_Project/s2_2022_b2_b3_b4_b8.tif'
path_2024 = '/content/drive/MyDrive/Deforestation_Project/s2_2024_b2_b3_b4_b8.tif'

print("Generating full-scene forest prediction mask for 2022...")
mask_2022, profile = predict_full_raster(path_2022)

print("Generating full-scene forest prediction mask for 2024...")
mask_2024, _ = predict_full_raster(path_2024)
print("Inference completed for both dates!")

In [ ]:
from scipy.ndimage import binary_opening

# 1. Post-Classification Comparison (PCC)
# Tree in 2022 (1) AND No Tree in 2024 (0)
raw_deforestation = (mask_2022 == 1) & (mask_2024 == 0)

# 2. Morphological noise cleanup (removes isolated pixel flickers)
cleaned_deforestation = binary_opening(raw_deforestation, structure=np.ones((3, 3))).astype(np.uint8)

# 3. Calculate summary metrics
pixel_area_ha = (10 * 10) / 10000.0  # 10m x 10m = 100 m² = 0.01 hectares
total_cleared_pixels = int(np.sum(cleaned_deforestation))
total_cleared_ha = total_cleared_pixels * pixel_area_ha

print(f"Total Deforested Pixels: {total_cleared_pixels}")
print(f"Estimated Forest Loss: {total_cleared_ha:.2f} hectares ({total_cleared_ha * 2.471:.2f} acres)")

In [ ]:
import geopandas as gpd
from shapely.geometry import shape
import rasterio.features

# Extract vector polygons directly from the raster mask using the geospatial transform
results = (
    {'properties': {'raster_val': int(v)}, 'geometry': s}
    for i, (s, v) in enumerate(
        rasterio.features.shapes(cleaned_deforestation, mask=cleaned_deforestation == 1, transform=profile['transform'])
    )
)

geoms = list(results)
if len(geoms) > 0:
    gdf = gpd.GeoDataFrame.from_features(geoms, crs=profile['crs'])

    # Calculate polygon-level metrics
    gdf['pixel_count'] = [int(shape(g).area / (profile['transform'][0] * abs(profile['transform'][4]))) for g in gdf['geometry']]
    # Calculate approximate area in hectares
    gdf['area_ha'] = (gdf.geometry.to_crs(epsg=3857).area / 10000.0).round(2)
    gdf['detection_date'] = "2022-2024"

    # Filter out tiny residual specks (< 0.2 hectares / 20 pixels)
    gdf_filtered = gdf[gdf['area_ha'] >= 0.2].copy()

    output_geojson = "/content/drive/MyDrive/Deforestation_Project/deforestation_alerts.geojson"
    gdf_filtered.to_file(output_geojson, driver="GeoJSON")
    print(f"Exported {len(gdf_filtered)} deforestation polygons to: {output_geojson}")
else:
    print("No change detected!")

In [ ]:
import matplotlib.pyplot as plt

# Read 2024 RGB for visual background
with rasterio.open(path_2024) as src:
    r = src.read(3)
    g = src.read(2)
    b = src.read(1)
    rgb_2024 = np.dstack([np.clip(r/3000.0, 0, 1), np.clip(g/3000.0, 0, 1), np.clip(b/3000.0, 0, 1)])

fig, ax = plt.subplots(1, 2, figsize=(16, 8))

ax[0].imshow(cleaned_deforestation, cmap='hot')
ax[0].set_title("U-Net Deforestation Mask (2022 vs 2024)", fontsize=13)
ax[0].axis('off')

ax[1].imshow(rgb_2024)
# Overlay the vectorized polygons in bright red
if len(geoms) > 0 and len(gdf_filtered) > 0:
    gdf_filtered.boundary.plot(ax=ax[1], color='red', linewidth=1.5)
ax[1].set_title("2024 Sentinel-2 with Detected Deforestation Polygons", fontsize=13)
ax[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt

def export_display_png(tif_path, output_png):
    with rasterio.open(tif_path) as src:
        # Read Red (3), Green (2), Blue (1)
        r = src.read(3).astype(np.float32)
        g = src.read(2).astype(np.float32)
        b = src.read(1).astype(np.float32)
        bounds = src.bounds

        # Normalize 0-3000 reflectance to [0, 1]
        rgb = np.dstack([
            np.clip(r / 3000.0, 0, 1),
            np.clip(g / 3000.0, 0, 1),
            np.clip(b / 3000.0, 0, 1)
        ])

        # Save as standard RGB PNG
        plt.imsave(output_png, rgb)
        print(f"Saved {output_png}")
        return [bounds.bottom, bounds.left, bounds.top, bounds.right]

path_2022 = '/content/drive/MyDrive/Deforestation_Project/s2_2022_b2_b3_b4_b8.tif'
path_2024 = '/content/drive/MyDrive/Deforestation_Project/s2_2024_b2_b3_b4_b8.tif'

bounds_2022 = export_display_png(path_2022, '/content/drive/MyDrive/Deforestation_Project/display_2022.png')
bounds_2024 = export_display_png(path_2024, '/content/drive/MyDrive/Deforestation_Project/display_2024.png')

print("Image bounds [min_lat, min_lon, max_lat, max_lon]:", bounds_2022)